# SurfaceGuard AI — Clean Consolidated Notebook (v7)

A fresh build, not an edit of the earlier v6 file, folding in the group-chat strategy points (MiT-B4, more augmentation, TTA) as generic, independently-implemented technique — nothing here is adapted from any notebook whose authorship wasn't confirmed as your own.

- **Phase 0** — shared utilities with the corrected per-class weighted metric, generic training loop (now with gradient accumulation), a richer augmentation set, 4-way TTA, min-area post-processing
- **Phase 1** — Model A (ResNet34-UNet, trained from scratch, `encoder_weights=None`) to satisfy the "built from scratch" requirement, plus W&B logging on every model
- **Phase 2** — Model B (ResNet34-UNet, ImageNet pretrained), Model C (SegFormer, MiT-B4 by default with gradient accumulation for T4 memory — set `SEGFORMER_VARIANT` below to switch to B0), Model D (DeepLabV3+ / ResNet50, the same architecture family as your own already-verified V6 run) — all four share one auxiliary-classification-head pattern, two-threshold gating, EMA
- **Phase 3** — ensemble, final submission, W&B run comparison pull

Every shape-sensitive block has a small dummy-tensor smoke test directly beneath it — **run those first on Kaggle, in order, before committing to any full training cell.** This was authored and reasoned through without GPU or dataset access on this end, so the smoke tests are the only verification it's had; they're deliberately placed to catch a shape or config error in seconds rather than after an hour of training.

Adjust `EPOCHS_*`, `BATCH_SIZE_*`, and `GRAD_ACCUM_*` per model to fit your remaining Kaggle GPU quota — the values below are conservative starting points, chosen to fit a T4, not fixed requirements.

**Rough time budget on a T4**, so you can plan which models to run in which session rather than finding out mid-run:

| Model | Default config | Rough time |
|---|---|---|
| A — scratch UNet | 20 epochs, ResNet34, batch 8 | ~1–1.5 hr |
| B — pretrained UNet | 15 epochs, ResNet34, batch 8 | ~45–70 min |
| C — SegFormer (B0 default) | 15 epochs, batch 8 | ~1–1.5 hr |
| C — SegFormer (B4, opt-in) | 6 epochs, batch 2 + grad accum | **~2.5–3 hr, measured directly on this dataset — not an estimate** |
| D — DeepLabV3+ | 12 epochs, ResNet50, batch 6 | ~1–1.5 hr |

With the B0 default throughout, all four models plus the ensemble/TTA pass fit comfortably inside a single Kaggle session. Switching Model C to B4 adds 2.5–3 hours on its own — plan that as its own session if you choose to use it, and don't expect to also fit Models A/B/D into the same run.

## Phase 0 — Setup

In [1]:
!pip install -q segmentation-models-pytorch transformers wandb opencv-python-headless

In [2]:
import os
import csv
import json
import math
import random
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import segmentation_models_pytorch as smp
from transformers import SegformerForSemanticSegmentation
import wandb

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

CLASSES = ["defect_a", "defect_b", "defect_c", "defect_d"]
HEIGHT = 128
WIDTH = 800

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = Path("/kaggle/input/competitions/industrial-surface-defect-detection/public")
TRAIN_IMAGE_DIR = DATA_ROOT / "train"
TEST_IMAGE_DIR = DATA_ROOT / "test"
TRAIN_CSV_PATH = DATA_ROOT / "train.csv"
SAMPLE_SUBMISSION_PATH = DATA_ROOT / "sample_submission.csv"

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = Path("/kaggle/working")

WANDB_ENTITY = "23f3003529-dl-genai-project"
WANDB_PROJECT = "23f3003529-t32026"

print("Device:", DEVICE)
print("Classes:", CLASSES)

Device: cuda
Classes: ['defect_a', 'defect_b', 'defect_c', 'defect_d']


In [4]:
assert TRAIN_CSV_PATH.exists(), (
    f"{TRAIN_CSV_PATH} not found. Run the next cell (os.listdir) to see what's actually "
    f"mounted under /kaggle/input, then fix DATA_ROOT above to match before continuing."
)
print("DATA_ROOT resolved correctly:", DATA_ROOT)

DATA_ROOT resolved correctly: /kaggle/input/competitions/industrial-surface-defect-detection/public


If `DATA_ROOT` doesn't match your Kaggle dataset mount path, update it here before running anything else — check with `!ls /kaggle/input`.

In [5]:
print(os.listdir("/kaggle/input"))

['competitions']


W&B login — uses the Kaggle Secrets-stored API key if `WANDB_API_KEY` is set and attached to this notebook (Add-ons → Secrets), otherwise falls back to an interactive `wandb.login()` prompt. That fallback is what hung the earlier run when the secret wasn't attached — fine if you're watching the session and can answer the prompt, but it will stall indefinitely on an unattended "Save & Run All" commit. Set the secret beforehand if you're running this unattended.

In [6]:
try:
    from kaggle_secrets import UserSecretsClient
    wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)
except Exception:
    wandb.login()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


## Data pipeline

RLE decode, mask lookup, and the train/validation split. The split is stratified on a per-image label signature (which of the four classes are present, or "clean") so rare classes like `defect_b` are proportionally represented in both splits.

In [7]:
def decode_rle(rle_string, shape):
    mask = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    if isinstance(rle_string, str) and rle_string.strip() not in ("", "-1"):
        tokens = rle_string.split()
        starts = np.asarray(tokens[0::2], dtype=np.int64) - 1
        lengths = np.asarray(tokens[1::2], dtype=np.int64)
        for start, length in zip(starts, lengths):
            mask[start:start + length] = 1
    return mask.reshape(shape, order="F")


def encode_rle(mask):
    pixels = mask.flatten(order="F")
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    if len(runs) == 0:
        return ""
    return " ".join(str(x) for x in runs)

In [8]:
_self_test_mask = np.zeros((HEIGHT, WIDTH), dtype=np.uint8)
_self_test_mask[10:15, 20:25] = 1
_self_test_mask[100:103, 700:720] = 1
_rle = encode_rle(_self_test_mask)
_decoded = decode_rle(_rle, (HEIGHT, WIDTH))
assert np.array_equal(_self_test_mask, _decoded), "RLE round-trip failed"
print("RLE round-trip OK")

RLE round-trip OK


In [9]:
train_df_raw = pd.read_csv(TRAIN_CSV_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

image_ids = sorted(train_df_raw["image_id"].astype(str).unique())
print("Unique training images:", len(image_ids))
print(train_df_raw.head())

Unique training images: 11393
     image_id defect_type                                           mask_rle  \
0  4S22I7SA34    defect_a  7255 6 7377 12 7505 12 7633 12 7762 11 7890 11...   
1  4S22I7SA34    defect_b                                                NaN   
2  4S22I7SA34    defect_c                                                NaN   
3  4S22I7SA34    defect_d                                                NaN   
4  DZ899111OE    defect_a                                                NaN   

   width  height  
0    800     128  
1    800     128  
2    800     128  
3    800     128  
4    800     128  


In [10]:
def build_mask_lookup(dataframe, image_ids, classes, height, width):
    lookup = {}
    grouped = dataframe.groupby("image_id")
    for image_id in tqdm(image_ids, desc="Decoding masks"):
        stack = np.zeros((len(classes), height, width), dtype=np.uint8)
        if image_id in grouped.groups:
            rows = grouped.get_group(image_id)
            for _, row in rows.iterrows():
                if row["defect_type"] in classes:
                    class_idx = classes.index(row["defect_type"])
                    stack[class_idx] = decode_rle(row["mask_rle"], (height, width))
        lookup[image_id] = stack
    return lookup


mask_lookup = build_mask_lookup(train_df_raw, image_ids, CLASSES, HEIGHT, WIDTH)

Decoding masks:   0%|          | 0/11393 [00:00<?, ?it/s]

In [11]:
def label_signature(image_id, lookup, classes):
    mask = lookup[image_id]
    present = [classes[i] for i in range(len(classes)) if mask[i].any()]
    return "clean" if len(present) == 0 else "+".join(present)


labels_df = pd.DataFrame({"image_id": image_ids})
labels_df["label_combo"] = labels_df["image_id"].apply(lambda x: label_signature(x, mask_lookup, CLASSES))

rare_combo_counts = labels_df["label_combo"].value_counts()
labels_df["stratify_key"] = labels_df["label_combo"].where(
    labels_df["label_combo"].map(rare_combo_counts) >= 2, "rare_other"
)

train_df, val_df = train_test_split(
    labels_df, test_size=0.20, random_state=SEED, stratify=labels_df["stratify_key"]
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train images:", len(train_df), "| Val images:", len(val_df))
print(train_df["label_combo"].value_counts().head(10))

Train images: 9114 | Val images: 2279
label_combo
clean                         4413
defect_c                      2484
defect_d                       657
defect_a                       481
defect_c+defect_d              478
defect_a+defect_c              172
defect_b                       121
defect_a+defect_d              109
defect_a+defect_c+defect_d      70
defect_b+defect_c               54
Name: count, dtype: int64


## Dataset, augmentation, weighted sampler

Augmentation set, in order: both flip axes, brightness/contrast jitter, light gaussian noise, a small translation (shift, not crop — keeps the full strip visible), and coarse dropout (cutout) applied to the image only. Cutout forces the model to not over-rely on any single patch of the strip rather than removing label information, so it's left off the mask. This is a heavier augmentation set than before, directly in response to wanting more regularization — applied in raw [0,1] pixel space, before per-image z-score normalization, so brightness/cutout operate on actual pixel intensities rather than already-normalized values.

The weighted sampler oversamples images containing rare classes (`defect_a`, `defect_b`), capped at 3x to avoid overfitting on a handful of repeated images.

In [12]:
def random_shift(image, mask, max_shift_frac=0.04):
    height, width = image.shape
    shift_y = random.randint(-int(height * max_shift_frac), int(height * max_shift_frac))
    shift_x = random.randint(-int(width * max_shift_frac), int(width * max_shift_frac))
    if shift_y == 0 and shift_x == 0:
        return image, mask
    image = np.roll(image, shift=(shift_y, shift_x), axis=(0, 1))
    mask = np.roll(mask, shift=(shift_y, shift_x), axis=(1, 2))
    return image, mask


def cutout(image, max_patches=2, patch_frac=0.08):
    height, width = image.shape
    image = image.copy()
    for _ in range(random.randint(1, max_patches)):
        patch_h = max(1, int(height * patch_frac * random.uniform(0.5, 1.5)))
        patch_w = max(1, int(width * patch_frac * random.uniform(0.5, 1.5)))
        top = random.randint(0, max(height - patch_h, 0))
        left = random.randint(0, max(width - patch_w, 0))
        image[top:top + patch_h, left:left + patch_w] = float(image.mean())
    return image


class SurfaceDefectDataset(Dataset):
    def __init__(self, dataframe, image_dir, mask_lookup, classes, height, width, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.mask_lookup = mask_lookup
        self.classes = classes
        self.height = height
        self.width = width
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_id = str(self.df.iloc[idx]["image_id"])
        image_path = self.image_dir / f"{image_id}.jpg"

        image = Image.open(image_path).convert("L")
        image = image.resize((self.width, self.height), Image.Resampling.BILINEAR)
        image = np.asarray(image, dtype=np.float32) / 255.0

        mask = self.mask_lookup[image_id].astype(np.float32).copy()

        if self.augment:
            if random.random() < 0.5:
                image = np.flip(image, axis=1)
                mask = np.flip(mask, axis=2)
            if random.random() < 0.5:
                image = np.flip(image, axis=0)
                mask = np.flip(mask, axis=1)
            if random.random() < 0.3:
                image, mask = random_shift(image, mask)
            if random.random() < 0.5:
                gain = random.uniform(0.85, 1.15)
                bias = random.uniform(-0.08, 0.08)
                image = np.clip(image * gain + bias, 0.0, 1.0)
            if random.random() < 0.3:
                noise = np.random.normal(0.0, 0.02, size=image.shape).astype(np.float32)
                image = np.clip(image + noise, 0.0, 1.0)
            if random.random() < 0.25:
                image = cutout(image)

        mean = float(image.mean())
        std = max(float(image.std()), 0.05)
        image = (image - mean) / std

        image_tensor = torch.from_numpy(np.ascontiguousarray(image)).unsqueeze(0).float()
        mask_tensor = torch.from_numpy(np.ascontiguousarray(mask)).float()
        return image_tensor, mask_tensor


class TestDefectDataset(Dataset):
    def __init__(self, image_ids, image_dir, height, width):
        self.image_ids = list(image_ids)
        self.image_dir = Path(image_dir)
        self.height = height
        self.width = width

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_path = self.image_dir / f"{image_id}.jpg"

        image = Image.open(image_path).convert("L")
        image = image.resize((self.width, self.height), Image.Resampling.BILINEAR)
        image = np.asarray(image, dtype=np.float32) / 255.0

        mean = float(image.mean())
        std = max(float(image.std()), 0.05)
        image = (image - mean) / std

        image_tensor = torch.from_numpy(np.ascontiguousarray(image)).unsqueeze(0).float()
        return image_id, image_tensor

In [13]:
_sample_ds = SurfaceDefectDataset(train_df.head(4), TRAIN_IMAGE_DIR, mask_lookup, CLASSES, HEIGHT, WIDTH, augment=True)
_img, _msk = _sample_ds[0]
assert _img.shape == (1, HEIGHT, WIDTH), _img.shape
assert _msk.shape == (len(CLASSES), HEIGHT, WIDTH), _msk.shape
print("Dataset smoke test OK —", _img.shape, _msk.shape)

Dataset smoke test OK — torch.Size([1, 128, 800]) torch.Size([4, 128, 800])


In [14]:
def build_class_image_counts(image_ids, mask_lookup, classes):
    counts = np.zeros(len(classes), dtype=np.int64)
    for image_id in image_ids:
        present = mask_lookup[image_id].any(axis=(1, 2))
        counts += present.astype(np.int64)
    return counts


def build_sample_weights(image_ids, mask_lookup, classes, class_image_counts, cap=3.0):
    weights = []
    n = len(image_ids)
    for image_id in image_ids:
        present_idx = np.where(mask_lookup[image_id].any(axis=(1, 2)))[0]
        if len(present_idx) == 0:
            weights.append(1.0)
        else:
            candidate_weights = [
                min(cap, np.sqrt(n / max(int(class_image_counts[c]), 1))) for c in present_idx
            ]
            weights.append(max(candidate_weights))
    return torch.as_tensor(weights, dtype=torch.double)


train_class_counts = build_class_image_counts(train_df["image_id"].astype(str), mask_lookup, CLASSES)
print("Train positive-image counts per class:", dict(zip(CLASSES, train_class_counts.tolist())))

sample_weights = build_sample_weights(train_df["image_id"].astype(str), mask_lookup, CLASSES, train_class_counts)
train_sampler = torch.utils.data.WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

Train positive-image counts per class: {'defect_a': 865, 'defect_b': 250, 'defect_c': 3280, 'defect_d': 1365}


In [15]:
def build_loaders(batch_size, num_workers=2):
    train_dataset = SurfaceDefectDataset(train_df, TRAIN_IMAGE_DIR, mask_lookup, CLASSES, HEIGHT, WIDTH, augment=True)
    val_dataset = SurfaceDefectDataset(val_df, TRAIN_IMAGE_DIR, mask_lookup, CLASSES, HEIGHT, WIDTH, augment=False)

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, sampler=train_sampler,
        num_workers=num_workers, pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
    )
    return train_loader, val_loader

## Losses

Hybrid BCE + Focal Tversky for segmentation (BCE anchors probability calibration, Tversky handles the class-imbalance-aware overlap objective — see the writeup above for why pure Tversky caused the threshold-saturation bug in the V3 run). A separate weighted BCE for the auxiliary image-level classification head.

In [16]:
def focal_tversky_loss(logits, targets, alpha=0.3, beta=0.7, gamma=0.75, eps=1e-6):
    probabilities = torch.sigmoid(logits.float())
    targets = targets.float()
    dims = (0, 2, 3)
    tp = (probabilities * targets).sum(dim=dims)
    fp = (probabilities * (1 - targets)).sum(dim=dims)
    fn = ((1 - probabilities) * targets).sum(dim=dims)
    tversky = (tp + eps) / (tp + alpha * fp + beta * fn + eps)
    return ((1 - tversky).clamp_min(0) ** gamma).mean()


def segmentation_loss(logits, targets):
    bce = nn.functional.binary_cross_entropy_with_logits(logits.float(), targets.float())
    tversky = focal_tversky_loss(logits, targets)
    return 0.5 * bce + 0.5 * tversky


def classification_loss(cls_logits, masks, pos_weight):
    image_labels = (masks.sum(dim=(-2, -1)) > 0).float()
    return nn.functional.binary_cross_entropy_with_logits(cls_logits.float(), image_labels, pos_weight=pos_weight)


CLASS_POS_WEIGHT = torch.as_tensor(
    [(len(train_df) - c) / max(c, 1) for c in train_class_counts], dtype=torch.float32, device=DEVICE
)
print("Classification pos_weight per class:", CLASS_POS_WEIGHT.tolist())

Classification pos_weight per class: [9.536416053771973, 35.45600128173828, 1.7786585092544556, 5.6769232749938965]


In [17]:
_dummy_logits = torch.randn(4, len(CLASSES), 16, 16)
_dummy_targets = (torch.rand(4, len(CLASSES), 16, 16) > 0.9).float()
_loss_value = segmentation_loss(_dummy_logits, _dummy_targets)
assert torch.isfinite(_loss_value)
print("Segmentation loss smoke test OK —", float(_loss_value))

_dummy_cls_logits = torch.randn(4, len(CLASSES))
_cls_loss_value = classification_loss(_dummy_cls_logits, _dummy_targets, CLASS_POS_WEIGHT.cpu())
assert torch.isfinite(_cls_loss_value)
print("Classification loss smoke test OK —", float(_cls_loss_value))

Segmentation loss smoke test OK — 0.8073322772979736
Classification loss smoke test OK — 8.962870597839355


## The corrected weighted competition metric

This is the single function every model's threshold selection and checkpointing goes through — defined once here so the per-class normalization bug from the V4/V5 run (dividing by the global pair-count total instead of each class's own count) can't silently reappear in a copy-pasted cell. `forward_fn` takes a batch of images and returns segmentation logits at full resolution — each model below supplies its own.

In [18]:
THRESHOLD_GRID = np.array([
    0.005, 0.01, 0.02, 0.03, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20,
    0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.98, 0.99
], dtype=np.float32)


def evaluate_weighted_metric(forward_fn, loader, thresholds, classes, height, width, device):
    num_t = len(thresholds)
    num_c = len(classes)
    pixels_per_pair = height * width

    pos_dice_sum = np.zeros((num_t, num_c))
    fp_pixels_sum = np.zeros((num_t, num_c))
    empty_correct_sum = np.zeros((num_t, num_c))
    pos_counts = np.zeros(num_c, dtype=np.int64)
    empty_counts = np.zeros(num_c, dtype=np.int64)

    threshold_tensor = torch.as_tensor(thresholds, device=device).view(-1, 1, 1, 1, 1)

    with torch.inference_mode():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
                logits = forward_fn(images)
            probabilities = torch.sigmoid(logits.float())

            targets = masks >= 0.5
            target_counts = targets.sum(dim=(-2, -1))
            positive = target_counts > 0
            empty = ~positive
            pos_counts += positive.sum(dim=0).cpu().numpy()
            empty_counts += empty.sum(dim=0).cpu().numpy()

            predictions = probabilities.unsqueeze(0) >= threshold_tensor
            predicted_counts = predictions.sum(dim=(-2, -1))
            intersections = (predictions & targets.unsqueeze(0)).sum(dim=(-2, -1))
            pair_dice = 2.0 * intersections.float() / (predicted_counts + target_counts.unsqueeze(0)).clamp_min(1)

            pos_dice_sum += (pair_dice * positive.unsqueeze(0)).sum(dim=1).cpu().numpy()
            fp_pixels_sum += (predicted_counts * empty.unsqueeze(0)).sum(dim=1).cpu().numpy()
            empty_correct_sum += ((predicted_counts == 0) & empty.unsqueeze(0)).sum(dim=1).cpu().numpy()

    contributions = np.zeros((num_t, num_c))
    for c in range(num_c):
        contributions[:, c] = (
            0.70 * pos_dice_sum[:, c] / pos_counts[c]
            - 0.20 * fp_pixels_sum[:, c] / (empty_counts[c] * pixels_per_pair)
            + 0.10 * empty_correct_sum[:, c] / empty_counts[c]
        )

    selected = contributions.argmax(axis=0)
    results = []
    for c, name in enumerate(classes):
        idx = int(selected[c])
        results.append({
            "class": name,
            "threshold": float(thresholds[idx]),
            "positive_pair_dice": float(pos_dice_sum[idx, c] / pos_counts[c]),
            "empty_fp_fraction": float(fp_pixels_sum[idx, c] / (empty_counts[c] * pixels_per_pair)),
            "empty_accuracy": float(empty_correct_sum[idx, c] / empty_counts[c]),
            "grid_edge": bool(idx == 0 or idx == num_t - 1),
            "weighted_score": float(contributions[idx, c]),
        })
        if results[-1]["grid_edge"]:
            print(f"WARNING: {name} threshold saturated at grid edge ({results[-1]['threshold']}) — widen THRESHOLD_GRID")
    return results, contributions

In [19]:
_dummy_forward = lambda x: torch.randn(x.shape[0], len(CLASSES), HEIGHT, WIDTH, device=x.device)
_dummy_loader = [(torch.randn(2, 1, HEIGHT, WIDTH), (torch.rand(2, len(CLASSES), HEIGHT, WIDTH) > 0.95).float()) for _ in range(3)]
_results, _ = evaluate_weighted_metric(_dummy_forward, _dummy_loader, THRESHOLD_GRID, CLASSES, HEIGHT, WIDTH, torch.device("cpu"))
assert len(_results) == len(CLASSES)
print("Weighted metric smoke test OK")
for r in _results:
    print(r)

Weighted metric smoke test OK
{'class': 'defect_a', 'threshold': 0.004999999888241291, 'positive_pair_dice': 0.09482090175151825, 'empty_fp_fraction': nan, 'empty_accuracy': nan, 'grid_edge': True, 'weighted_score': nan}
{'class': 'defect_b', 'threshold': 0.004999999888241291, 'positive_pair_dice': 0.09371851632992427, 'empty_fp_fraction': nan, 'empty_accuracy': nan, 'grid_edge': True, 'weighted_score': nan}
{'class': 'defect_c', 'threshold': 0.004999999888241291, 'positive_pair_dice': 0.09524636964003245, 'empty_fp_fraction': nan, 'empty_accuracy': nan, 'grid_edge': True, 'weighted_score': nan}
{'class': 'defect_d', 'threshold': 0.004999999888241291, 'positive_pair_dice': 0.09482747316360474, 'empty_fp_fraction': nan, 'empty_accuracy': nan, 'grid_edge': True, 'weighted_score': nan}


/tmp/ipykernel_23/2832607211.py:49: RuntimeWarning: invalid value encountered in divide
  - 0.20 * fp_pixels_sum[:, c] / (empty_counts[c] * pixels_per_pair)
/tmp/ipykernel_23/2832607211.py:50: RuntimeWarning: invalid value encountered in divide
  + 0.10 * empty_correct_sum[:, c] / empty_counts[c]
/tmp/ipykernel_23/2832607211.py:61: RuntimeWarning: invalid value encountered in scalar divide
  "empty_fp_fraction": float(fp_pixels_sum[idx, c] / (empty_counts[c] * pixels_per_pair)),
/tmp/ipykernel_23/2832607211.py:62: RuntimeWarning: invalid value encountered in scalar divide
  "empty_accuracy": float(empty_correct_sum[idx, c] / empty_counts[c]),


## EMA and the generic training loop

Every model below reuses this one training function — same checkpointing rule (true weighted metric, not raw Dice@0.5), same W&B logging shape, same EMA handling. At the end of training the function compares the final raw weights against the EMA-smoothed weights on validation and keeps whichever scores higher, since training has been visibly noisy epoch-to-epoch (see the V5 defect_a curve in the writeup above).

In [20]:
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    def update(self, model):
        with torch.no_grad():
            for k, v in model.state_dict().items():
                if v.dtype.is_floating_point:
                    self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
                else:
                    self.shadow[k] = v.detach().clone()

In [21]:
def run_validation(model, forward_single, val_loader, device, state_dict=None):
    original_state = None
    if state_dict is not None:
        original_state = copy.deepcopy(model.state_dict())
        model.load_state_dict(state_dict, strict=True)
    model.eval()

    def fwd(images):
        output = forward_single(model, images)
        return output[0] if isinstance(output, tuple) else output

    results, contributions = evaluate_weighted_metric(fwd, val_loader, THRESHOLD_GRID, CLASSES, HEIGHT, WIDTH, device)

    if original_state is not None:
        model.load_state_dict(original_state, strict=True)
    model.train()
    return results, contributions


def train_model(
    model, forward_single, train_loader, val_loader, optimizer, scheduler,
    epochs, device, checkpoint_path, run_name, has_aux_head=False,
    cls_pos_weight=None, aux_loss_weight=0.3, use_ema=True, ema_decay=0.999,
    grad_accum_steps=1, run_config=None,
):
    wandb.init(project=WANDB_PROJECT, name=run_name, config=run_config or {}, reinit=True)

    model.to(device)
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
    ema = EMA(model, decay=ema_decay) if use_ema else None

    best_score = -1.0
    best_state = None
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        batches = 0
        optimizer.zero_grad(set_to_none=True)

        for step, (images, masks) in enumerate(tqdm(train_loader, desc=f"{run_name} | epoch {epoch}/{epochs}")):
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
                output = forward_single(model, images)
                if has_aux_head:
                    seg_logits, cls_logits = output
                    loss = segmentation_loss(seg_logits, masks) + aux_loss_weight * classification_loss(
                        cls_logits, masks, cls_pos_weight
                    )
                else:
                    seg_logits = output
                    loss = segmentation_loss(seg_logits, masks)
                loss_to_backward = loss / grad_accum_steps

            scaler.scale(loss_to_backward).backward()

            is_last_in_accum = (step + 1) % grad_accum_steps == 0
            is_last_batch = (step + 1) == len(train_loader)
            if is_last_in_accum or is_last_batch:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if scheduler is not None:
                    scheduler.step()
                if ema is not None:
                    ema.update(model)

            running_loss += float(loss.item())
            batches += 1

        train_loss = running_loss / max(batches, 1)
        results, _ = run_validation(model, forward_single, val_loader, device)
        class_scores = {r["class"]: r["weighted_score"] for r in results}
        mean_score = float(np.mean(list(class_scores.values())))

        log_payload = {"epoch": epoch, "train_loss": train_loss, "val_weighted_score": mean_score}
        log_payload.update({f"{k}_weighted_score": v for k, v in class_scores.items()})
        wandb.log(log_payload)
        history.append(log_payload)
        print(f"{run_name} | epoch {epoch}/{epochs} | train_loss {train_loss:.5f} | val_weighted_score {mean_score:.5f}")

        if mean_score > best_score:
            best_score = mean_score
            best_state = copy.deepcopy(model.state_dict())

    final_best_state = best_state
    final_best_score = best_score

    if ema is not None:
        ema_results, _ = run_validation(model, forward_single, val_loader, device, state_dict=ema.shadow)
        ema_score = float(np.mean([r["weighted_score"] for r in ema_results]))
        print(f"{run_name} | best raw score {best_score:.5f} | EMA score {ema_score:.5f}")
        wandb.log({"ema_val_weighted_score": ema_score})
        if ema_score > final_best_score:
            final_best_state = copy.deepcopy(ema.shadow)
            final_best_score = ema_score

    torch.save({
        "model_state_dict": final_best_state,
        "weighted_score": final_best_score,
        "classes": CLASSES, "height": HEIGHT, "width": WIDTH,
        "run_name": run_name,
    }, checkpoint_path)
    wandb.finish()

    return pd.DataFrame(history), final_best_score

## Model A — ResNet34-UNet, built from scratch

`encoder_weights=None` — no ImageNet transfer, fully random initialization, trained only on this dataset. This satisfies the "one model built from scratch" requirement directly, and gives a clean, directly-comparable baseline against Model B (identical architecture, pretrained) for the report's transfer-learning discussion.

`aux_params` adds a built-in image-level classification head (documented `segmentation_models_pytorch` feature) — `model(x)` returns `(seg_logits, cls_logits)`.

In [22]:
def build_unet(encoder_name, encoder_weights, num_classes):
    return smp.Unet(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=1,
        classes=num_classes,
        activation=None,
        aux_params=dict(pooling="avg", dropout=0.3, activation=None, classes=num_classes),
    )


def unet_forward(model, images):
    seg_logits, cls_logits = model(images)
    return seg_logits, cls_logits

In [23]:
_dummy_model = build_unet("resnet34", None, len(CLASSES))
_dummy_input = torch.randn(2, 1, HEIGHT, WIDTH)
_seg_out, _cls_out = unet_forward(_dummy_model, _dummy_input)
assert _seg_out.shape == (2, len(CLASSES), HEIGHT, WIDTH), _seg_out.shape
assert _cls_out.shape == (2, len(CLASSES)), _cls_out.shape
print("Unet + aux head smoke test OK —", _seg_out.shape, _cls_out.shape)
del _dummy_model, _dummy_input, _seg_out, _cls_out

Unet + aux head smoke test OK — torch.Size([2, 4, 128, 800]) torch.Size([2, 4])


In [24]:
EPOCHS_SCRATCH = 20
BATCH_SIZE_SCRATCH = 8

train_loader_cnn, val_loader_cnn = build_loaders(batch_size=BATCH_SIZE_SCRATCH)

model_scratch = build_unet("resnet34", None, len(CLASSES)).to(DEVICE)
optimizer_scratch = torch.optim.AdamW(model_scratch.parameters(), lr=3e-4, weight_decay=1e-4)
total_steps_scratch = len(train_loader_cnn) * EPOCHS_SCRATCH
scheduler_scratch = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_scratch, T_max=total_steps_scratch)

history_scratch, score_scratch = train_model(
    model_scratch, unet_forward, train_loader_cnn, val_loader_cnn,
    optimizer_scratch, scheduler_scratch, EPOCHS_SCRATCH, DEVICE,
    CHECKPOINT_DIR / "model_a_scratch.pt", run_name="model-a-resnet34-scratch",
    has_aux_head=True, cls_pos_weight=CLASS_POS_WEIGHT,
    run_config={"encoder": "resnet34", "pretrained": False, "epochs": EPOCHS_SCRATCH, "role": "built_from_scratch"},
)
print("Model A (scratch) best weighted score:", score_scratch)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261111_182318-bkk084oc
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model-a-resnet34-scratch
wandb: ⭐️ View project at https://wandb.ai/23f3003529-dl-genai-project/23f3003529-t32026
wandb: 🚀 View run at https://wandb.ai/23f3003529-dl-genai-project/23f3003529-t32026/runs/bkk084oc


model-a-resnet34-scratch | epoch 1/20:   0%|          | 0/1139 [00:00<?, ?it/s]

model-a-resnet34-scratch | epoch 1/20 | train_loss 0.96360 | val_weighted_score 0.17183


model-a-resnet34-scratch | epoch 2/20:   0%|          | 0/1139 [00:00<?, ?it/s]

model-a-resnet34-scratch | epoch 2/20 | train_loss 0.89304 | val_weighted_score 0.20404


model-a-resnet34-scratch | epoch 3/20:   0%|          | 0/1139 [00:00<?, ?it/s]

... (33 outputs left out)

model-a-resnet34-scratch | epoch 20/20:   0%|          | 0/1139 [00:00<?, ?it/s]

model-a-resnet34-scratch | epoch 20/20 | train_loss 0.63511 | val_weighted_score 0.35208
model-a-resnet34-scratch | best raw score 0.35315 | EMA score 0.35263


wandb: updating run metadata
wandb: 
wandb: Run history:
wandb: defect_a_weighted_score ▁▂▃▃▄▅▆▅▆▇▇▇▇▇██████
wandb: defect_b_weighted_score ▁▁▄▄▃▅▆▆▆▆▆▇▇▇██████
wandb: defect_c_weighted_score ▁▃▃▃▄▅▅▆▅▇▇▇▇▇▇█████
wandb: defect_d_weighted_score ▁▃▃▃▅▅▆▆▆▇▇▇████████
wandb:  ema_val_weighted_score ▁
wandb:                   epoch ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
wandb:              train_loss █▆▆▅▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁▁
wandb:      val_weighted_score ▁▂▃▃▄▅▆▆▆▇▇▇▇▇██████
wandb: 
wandb: Run summary:
wandb: defect_a_weighted_score 0.26412
wandb: defect_b_weighted_score 0.31197
wandb: defect_c_weighted_score 0.36501
wandb: defect_d_weighted_score 0.46723
wandb:  ema_val_weighted_score 0.35263
wandb:                   epoch 20
wandb:              train_loss 0.63511
wandb:      val_weighted_score 0.35208
wandb: 
wandb: 🚀 View run model-a-resnet34-scratch at: https://wandb.ai/23f3003529-dl-genai-project/23f3003529-t32026/runs/bkk084oc
wandb: ⭐️ View project at: https://wandb.ai/23f3003529-dl-genai-project/23f3003

Model A (scratch) best weighted score: 0.35314729815336987


## Model B — ResNet34-UNet, ImageNet pretrained

Same architecture and training recipe as Model A — only `encoder_weights` differs. This isolates the transfer-learning effect cleanly for the report: same data, same augmentation, same loss, same schedule, same classification head. Satisfies "one pretrained model" directly.

In [25]:
EPOCHS_PRETRAINED = 15
BATCH_SIZE_PRETRAINED = 8

model_pretrained = build_unet("resnet34", "imagenet", len(CLASSES)).to(DEVICE)
optimizer_pretrained = torch.optim.AdamW(model_pretrained.parameters(), lr=1e-4, weight_decay=1e-4)
total_steps_pretrained = len(train_loader_cnn) * EPOCHS_PRETRAINED
scheduler_pretrained = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_pretrained, T_max=total_steps_pretrained)

history_pretrained, score_pretrained = train_model(
    model_pretrained, unet_forward, train_loader_cnn, val_loader_cnn,
    optimizer_pretrained, scheduler_pretrained, EPOCHS_PRETRAINED, DEVICE,
    CHECKPOINT_DIR / "model_b_pretrained.pt", run_name="model-b-resnet34-imagenet",
    has_aux_head=True, cls_pos_weight=CLASS_POS_WEIGHT,
    run_config={"encoder": "resnet34", "pretrained": True, "epochs": EPOCHS_PRETRAINED, "role": "pretrained"},
)
print("Model B (pretrained) best weighted score:", score_pretrained)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261111_191555-yjpm3fbl
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model-b-resnet34-imagenet
wandb: ⭐️ View project at https://wandb.ai/23f3003529-dl-genai-project/23f3003529-t32026
wandb: 🚀 View run at https://wandb.ai/23f3003529-dl-genai-project/23f3003529-t32026/runs/yjpm3fbl


model-b-resnet34-imagenet | epoch 1/15:   0%|          | 0/1139 [00:00<?, ?it/s]

model-b-resnet34-imagenet | epoch 1/15 | train_loss 0.92083 | val_weighted_score 0.22772


... (26 outputs left out)

model-b-resnet34-imagenet | epoch 15/15:   0%|          | 0/1139 [00:00<?, ?it/s]

model-b-resnet34-imagenet | epoch 15/15 | train_loss 0.43986 | val_weighted_score 0.43049
model-b-resnet34-imagenet | best raw score 0.43049 | EMA score 0.42841


wandb: updating run metadata
wandb: uploading history steps 15-15, summary, console lines 17-17
wandb: 
wandb: Run history:
wandb: defect_a_weighted_score ▁▅▇▇▆▇▇▇▇▇█████
wandb: defect_b_weighted_score ▁▄▄▄▄▇▆█▇▇█████
wandb: defect_c_weighted_score ▁▄▄▅▆▆▆▇▇██████
wandb: defect_d_weighted_score ▁▃▄▅▇▆▇▇▆██▇███
wandb:  ema_val_weighted_score ▁
wandb:                   epoch ▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
wandb:              train_loss █▅▅▄▄▃▃▃▂▂▂▁▁▁▁
wandb:      val_weighted_score ▁▄▅▅▆▇▆▇▇██████
wandb: 
wandb: Run summary:
wandb: defect_a_weighted_score 0.33811
wandb: defect_b_weighted_score 0.35372
wandb: defect_c_weighted_score 0.47037
wandb: defect_d_weighted_score 0.55977
wandb:  ema_val_weighted_score 0.42841
wandb:                   epoch 15
wandb:              train_loss 0.43986
wandb:      val_weighted_score 0.43049
wandb: 
wandb: 🚀 View run model-b-resnet34-imagenet at: https://wandb.ai/23f3003529-dl-genai-project/23f3003529-t32026/runs/yjpm3fbl
wandb: ⭐️ View project at: https://wandb.ai/23

Model B (pretrained) best weighted score: 0.43049223173640516


In [26]:
print(f"Transfer-learning effect: {score_pretrained - score_scratch:+.4f} weighted-score points from ImageNet pretraining alone")

Transfer-learning effect: +0.0773 weighted-score points from ImageNet pretraining alone


## Model C — SegFormer, MiT-B0 by default

`SEGFORMER_VARIANT` defaults to **`"b0"`**, reversed from the earlier draft of this notebook. A real timed run on this exact dataset came back at roughly **26 minutes per epoch for MiT-B4** at the batch size this GPU's memory allows — a 6-epoch run alone took 156 minutes of training, plus data/calibration/TTA overhead on top. Twelve epochs of B4, as this notebook originally defaulted to, would be 5+ hours for that one model alone, before Models A, B, and D even start. B0 is roughly 4x fewer encoder parameters and was already proven to train in reasonable time in your own earlier run (0.595 on the leaderboard). Set `SEGFORMER_VARIANT = "b4"` below only if you have 6+ hours of uninterrupted GPU quota specifically to spend on this one model — it is not the default for a reason.

**Gradient accumulation does not reduce wall-clock time.** `GRAD_ACCUM_SEGFORMER` only matters for B4: it keeps the effective batch size reasonable without running out of memory, by stepping the optimizer every few micro-batches instead of every one — but the same number of forward/backward passes still run every epoch either way. It buys memory headroom, not speed. If B4 is too slow, lowering `EPOCHS_SEGFORMER` is the only lever that actually helps; accumulation won't.

The classification head here is built by pooling the segmentation logits themselves (`torch.amax` over the spatial dimensions, per class) rather than tapping into the encoder's internal hidden states. This is a deliberate simplification: it only depends on `SegformerForSemanticSegmentation(...).logits`, the exact call already proven working in your own executed runs — avoiding a dependency on internal API details that can't be verified without running this on real hardware.

Discriminative learning rates: the decode head's final classifier layer was reinitialized from ADE20k's 150 classes down to 4 (`ignore_mismatched_sizes=True`), so it needs a higher LR than the pretrained encoder to catch up within a short fine-tune.

In [27]:
class SegFormerWithClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.base = SegformerForSemanticSegmentation.from_pretrained(
            model_name, num_labels=num_classes, ignore_mismatched_sizes=True
        )
        self.classifier_head = nn.Sequential(
            nn.Linear(num_classes, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_classes),
        )

    def forward(self, pixel_values, target_size):
        seg_logits = self.base(pixel_values=pixel_values).logits
        seg_logits = nn.functional.interpolate(seg_logits, size=target_size, mode="bilinear", align_corners=False)
        pooled = torch.amax(seg_logits, dim=(-2, -1))
        cls_logits = self.classifier_head(pooled)
        return seg_logits, cls_logits


def segformer_forward(model, images):
    rgb_images = images.repeat(1, 3, 1, 1)
    return model(rgb_images, target_size=(HEIGHT, WIDTH))

In [28]:
SEGFORMER_VARIANT = "b0"
SEGFORMER_MODEL_NAMES = {
    "b0": "nvidia/segformer-b0-finetuned-ade-512-512",
    "b4": "nvidia/segformer-b4-finetuned-ade-512-512",
}
SEGFORMER_MODEL_NAME = SEGFORMER_MODEL_NAMES[SEGFORMER_VARIANT]
print("SegFormer variant:", SEGFORMER_VARIANT, "|", SEGFORMER_MODEL_NAME)

SegFormer variant: b0 | nvidia/segformer-b0-finetuned-ade-512-512


In [29]:
_dummy_segformer = SegFormerWithClassifier(SEGFORMER_MODEL_NAME, len(CLASSES))
_dummy_input = torch.randn(1, 1, HEIGHT, WIDTH)
_seg_out, _cls_out = segformer_forward(_dummy_segformer, _dummy_input)
assert _seg_out.shape == (1, len(CLASSES), HEIGHT, WIDTH), _seg_out.shape
assert _cls_out.shape == (1, len(CLASSES)), _cls_out.shape
print("SegFormer + pooled-logit classifier smoke test OK —", _seg_out.shape, _cls_out.shape)
del _dummy_segformer, _dummy_input, _seg_out, _cls_out

config.json:   0%|          | 0.00/6.88k [00:00<?, ?B/s]

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `150`.


model.safetensors: reconstructing file:   0%|          |  0.00B / 15.0MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b0-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([4])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([4, 256, 1, 1])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


SegFormer + pooled-logit classifier smoke test OK — torch.Size([1, 4, 128, 800]) torch.Size([1, 4])


If you do switch to B4 and the smoke test runs out of memory even at batch size 1, that's a signal to drop back to `"b0"` — better to find that out in a few seconds than after the first training epoch. `EPOCHS_SEGFORMER` below is set per-variant: B0 trains fast enough to afford more epochs; B4's real measured cost (above) means 6 is already a 2.5+ hour commitment, and the timed run this was calibrated against had already plateaued by epoch 6 (its calibration loss improved only marginally from epoch 4 onward) — more epochs on B4 is not obviously worth the extra time.

In [30]:
EPOCHS_SEGFORMER = 6 if SEGFORMER_VARIANT == "b4" else 15
BATCH_SIZE_SEGFORMER = 2 if SEGFORMER_VARIANT == "b4" else 8
GRAD_ACCUM_SEGFORMER = 4 if SEGFORMER_VARIANT == "b4" else 1

train_loader_segformer, val_loader_segformer = build_loaders(batch_size=BATCH_SIZE_SEGFORMER)

model_segformer = SegFormerWithClassifier(SEGFORMER_MODEL_NAME, len(CLASSES)).to(DEVICE)

optimizer_segformer = torch.optim.AdamW([
    {"params": model_segformer.base.segformer.parameters(), "lr": 3e-5},
    {"params": model_segformer.base.decode_head.parameters(), "lr": 3e-4},
    {"params": model_segformer.classifier_head.parameters(), "lr": 3e-4},
], weight_decay=1e-4)

optimizer_steps_per_epoch = math.ceil(len(train_loader_segformer) / GRAD_ACCUM_SEGFORMER)
total_steps_segformer = optimizer_steps_per_epoch * EPOCHS_SEGFORMER
warmup_steps_segformer = optimizer_steps_per_epoch

def segformer_lr_lambda(step):
    if step < warmup_steps_segformer:
        return step / max(1, warmup_steps_segformer)
    progress = (step - warmup_steps_segformer) / max(1, total_steps_segformer - warmup_steps_segformer)
    return max(0.0, 1.0 - progress)

scheduler_segformer = torch.optim.lr_scheduler.LambdaLR(optimizer_segformer, segformer_lr_lambda)

history_segformer, score_segformer = train_model(
    model_segformer, segformer_forward, train_loader_segformer, val_loader_segformer,
    optimizer_segformer, scheduler_segformer, EPOCHS_SEGFORMER, DEVICE,
    CHECKPOINT_DIR / "model_c_segformer.pt", run_name=f"model-c-segformer-{SEGFORMER_VARIANT}",
    has_aux_head=True, cls_pos_weight=CLASS_POS_WEIGHT, grad_accum_steps=GRAD_ACCUM_SEGFORMER,
    run_config={"encoder": f"segformer-{SEGFORMER_VARIANT}", "pretrained": True, "epochs": EPOCHS_SEGFORMER, "role": "additional_choice"},
)
print("Model C (SegFormer) best weighted score:", score_segformer)

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `150`.


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b0-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([4])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([4, 256, 1, 1])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261111_195519-ez1xryf3
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model-c-segfo

model-c-segformer-b0 | epoch 1/15:   0%|          | 0/1139 [00:00<?, ?it/s]

model-c-segformer-b0 | epoch 1/15 | train_loss 0.98744 | val_weighted_score 0.23481


model-c-segformer-b0 | epoch 2/15:   0%|          | 0/1139 [00:00<?, ?it/s]

... (30 outputs left out)

wandb: updating run metadata


model-c-segformer-b0 | best raw score 0.30409 | EMA score 0.30323


wandb: 
wandb: Run history:
wandb: defect_a_weighted_score ▁▁▁▁▁▂▂▇▆█▆▄▄▃▃
wandb: defect_b_weighted_score ███▇▇█▆█▇██▁▇▆▆
wandb: defect_c_weighted_score ▁▃▅▆▆▆▇▇▇▇█████
wandb: defect_d_weighted_score ▁▄▅▆▇▇▇████████
wandb:  ema_val_weighted_score ▁
wandb:                   epoch ▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
wandb:              train_loss █▄▃▂▂▂▂▂▁▁▁▁▁▁▁
wandb:      val_weighted_score ▁▃▅▆▆▇▇▇▇██████
wandb: 
wandb: Run summary:
wandb: defect_a_weighted_score 0.10215
wandb: defect_b_weighted_score 0.09991
wandb: defect_c_weighted_score 0.44563
wandb: defect_d_weighted_score 0.56264
wandb:  ema_val_weighted_score 0.30323
wandb:                   epoch 15
wandb:              train_loss 0.5119
wandb:      val_weighted_score 0.30258
wandb: 
wandb: 🚀 View run model-c-segformer-b0 at: https://wandb.ai/23f3003529-dl-genai-project/23f3003529-t32026/runs/ez1xryf3
wandb: ⭐️ View project at: https://wandb.ai/23f3003529-dl-genai-project/23f3003529-t32026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact f

Model C (SegFormer) best weighted score: 0.30408666234195814


## Model D — DeepLabV3+, ResNet50 (bonus ensemble diversity)

Not required by the guideline (three models is the floor) — this is the same architecture family as your own already-executed, already-verified V6 run, which is why it's the chosen fourth member here rather than something untested: atrous/dilated convolutions give it a different receptive-field behavior than both the UNet decoder and the transformer, which is what makes it useful for the ensemble rather than redundant with Models A/B/C.

In [31]:
def build_deeplabv3plus(encoder_name, encoder_weights, num_classes):
    return smp.DeepLabV3Plus(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=1,
        classes=num_classes,
        activation=None,
        aux_params=dict(pooling="avg", dropout=0.3, activation=None, classes=num_classes),
    )

In [32]:
_dummy_model = build_deeplabv3plus("resnet50", "imagenet", len(CLASSES))
_dummy_model.eval()
_dummy_input = torch.randn(1, 1, HEIGHT, WIDTH)
_seg_out, _cls_out = unet_forward(_dummy_model, _dummy_input)
assert _seg_out.shape == (1, len(CLASSES), HEIGHT, WIDTH), _seg_out.shape
assert _cls_out.shape == (1, len(CLASSES)), _cls_out.shape
print("DeepLabV3+ smoke test OK —", _seg_out.shape, _cls_out.shape)
del _dummy_model, _dummy_input, _seg_out, _cls_out

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

DeepLabV3+ smoke test OK — torch.Size([1, 4, 128, 800]) torch.Size([1, 4])


In [33]:
EPOCHS_DEEPLAB = 12
BATCH_SIZE_DEEPLAB = 6

train_loader_deeplab, val_loader_deeplab = build_loaders(batch_size=BATCH_SIZE_DEEPLAB)

model_deeplab = build_deeplabv3plus("resnet50", "imagenet", len(CLASSES)).to(DEVICE)
optimizer_deeplab = torch.optim.AdamW(model_deeplab.parameters(), lr=1e-4, weight_decay=1e-4)
total_steps_deeplab = len(train_loader_deeplab) * EPOCHS_DEEPLAB
scheduler_deeplab = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_deeplab, T_max=total_steps_deeplab)

history_deeplab, score_deeplab = train_model(
    model_deeplab, unet_forward, train_loader_deeplab, val_loader_deeplab,
    optimizer_deeplab, scheduler_deeplab, EPOCHS_DEEPLAB, DEVICE,
    CHECKPOINT_DIR / "model_d_deeplabv3plus.pt", run_name="model-d-deeplabv3plus-resnet50",
    has_aux_head=True, cls_pos_weight=CLASS_POS_WEIGHT,
    run_config={"encoder": "resnet50", "decoder": "deeplabv3plus", "pretrained": True, "epochs": EPOCHS_DEEPLAB, "role": "bonus_ensemble_member"},
)
print("Model D (DeepLabV3+) best weighted score:", score_deeplab)

wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261111_202430-n5zvxbny
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model-d-deeplabv3plus-resnet50
wandb: ⭐️ View project at https://wandb.ai/23f3003529-dl-genai-project/23f3003529-t32026
wandb: 🚀 View run at https://wandb.ai/23f3003529-dl-genai-project/23f3003529-t32026/runs/n5zvxbny


model-d-deeplabv3plus-resnet50 | epoch 1/12:   0%|          | 0/1519 [00:00<?, ?it/s]

model-d-deeplabv3plus-resnet50 | epoch 1/12 | train_loss 0.84273 | val_weighted_score 0.26059


model-d-deeplabv3plus-resnet50 | epoch 2/12:   0%|          | 0/1519 [00:00<?, ?it/s]

model-d-deeplabv3plus-resnet50 | epoch 2/12 | train_loss 0.71744 | val_weighted_score 0.34950


model-d-deeplabv3plus-resnet50 | epoch 3/12:   0%|          | 0/1519 [00:00<?, ?it/s]

... (17 outputs left out)

model-d-deeplabv3plus-resnet50 | epoch 12/12:   0%|          | 0/1519 [00:00<?, ?it/s]

model-d-deeplabv3plus-resnet50 | epoch 12/12 | train_loss 0.46523 | val_weighted_score 0.42389
model-d-deeplabv3plus-resnet50 | best raw score 0.42389 | EMA score 0.42435


wandb: updating run metadata
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb: defect_a_weighted_score ▁▅▆▇▇▇█▇████
wandb: defect_b_weighted_score ▁▅▆▇▇█▇▇████
wandb: defect_c_weighted_score ▁▃▄▅▆▆▇█▇███
wandb: defect_d_weighted_score ▁▅▄▅▅▇▇▇█▇██
wandb:  ema_val_weighted_score ▁
wandb:                   epoch ▁▂▂▃▄▄▅▅▆▇▇█
wandb:              train_loss █▆▅▄▃▃▂▂▂▁▁▁
wandb:      val_weighted_score ▁▅▅▆▆▇▇▇████
wandb: 
wandb: Run summary:
wandb: defect_a_weighted_score 0.32447
wandb: defect_b_weighted_score 0.32566
wandb: defect_c_weighted_score 0.47469
wandb: defect_d_weighted_score 0.57073
wandb:  ema_val_weighted_score 0.42435
wandb:                   epoch 12
wandb:              train_loss 0.46523
wandb:      val_weighted_score 0.42389
wandb: 
wandb: 🚀 View run model-d-deeplabv3plus-resnet50 at: https://wandb.ai/23f3003529-dl-genai-project/23f3003529-t32026/runs/n5zvxbny
wandb: ⭐️ View project at: https://wandb.ai/23f3003529-dl-genai-projec

Model D (DeepLabV3+) best weighted score: 0.4243488266295803


## Phase 3 — Ensemble, two-threshold gating, final submission

Loads all four checkpoints, runs 4-way flip TTA (identity, horizontal, vertical, both — matching the training-time augmentation) per model, and blends probabilities across models. Starting blend weights below favor SegFormer slightly (it's the strongest single model so far) and the scratch model slightly less (expected to underperform the pretrained ones) — these are starting points, not tuned values. If you have time, sweep them on validation using the same weighted-metric machinery before finalizing.

In [34]:
def load_checkpoint_model(build_fn, checkpoint_path, forward_fn, weight=1.0):
    model = build_fn().to(DEVICE)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.eval()
    print(f"Loaded {checkpoint_path.name} | weighted_score={checkpoint['weighted_score']:.5f}")
    return (forward_fn, model, weight)


model_specs = [
    load_checkpoint_model(lambda: build_unet("resnet34", None, len(CLASSES)), CHECKPOINT_DIR / "model_a_scratch.pt", unet_forward, weight=0.8),
    load_checkpoint_model(lambda: build_unet("resnet34", "imagenet", len(CLASSES)), CHECKPOINT_DIR / "model_b_pretrained.pt", unet_forward, weight=1.0),
    load_checkpoint_model(lambda: SegFormerWithClassifier(SEGFORMER_MODEL_NAME, len(CLASSES)), CHECKPOINT_DIR / "model_c_segformer.pt", segformer_forward, weight=1.2),
    load_checkpoint_model(lambda: build_deeplabv3plus("resnet50", "imagenet", len(CLASSES)), CHECKPOINT_DIR / "model_d_deeplabv3plus.pt", unet_forward, weight=1.0),
]

Loaded model_a_scratch.pt | weighted_score=0.35315


[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `150`.


Loaded model_b_pretrained.pt | weighted_score=0.43049


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b0-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([4])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([4, 256, 1, 1])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Loaded model_c_segformer.pt | weighted_score=0.30409
Loaded model_d_deeplabv3plus.pt | weighted_score=0.42435


In [35]:
FLIP_VARIANTS = [None, (-1,), (-2,), (-2, -1)]


def apply_flip(tensor, dims):
    if dims is None:
        return tensor
    return torch.flip(tensor, dims=list(dims))


def predict_ensemble_tta(images, model_specs, device):
    seg_prob_sum = None
    cls_prob_sum = None
    total_weight = 0.0

    for forward_single, model, weight in model_specs:
        for dims in FLIP_VARIANTS:
            flipped = apply_flip(images, dims)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
                seg_logits, cls_logits = forward_single(model, flipped)
            seg_logits = apply_flip(seg_logits, dims)
            seg_probs = torch.sigmoid(seg_logits.float())
            cls_probs = torch.sigmoid(cls_logits.float())

            if seg_prob_sum is None:
                seg_prob_sum = torch.zeros_like(seg_probs)
                cls_prob_sum = torch.zeros_like(cls_probs)
            seg_prob_sum = seg_prob_sum + weight * seg_probs
            cls_prob_sum = cls_prob_sum + weight * cls_probs
        total_weight += weight * len(FLIP_VARIANTS)

    return seg_prob_sum / total_weight, cls_prob_sum / total_weight

In [36]:
_dummy_input = torch.randn(1, 1, HEIGHT, WIDTH, device=DEVICE)
_seg_probs, _cls_probs = predict_ensemble_tta(_dummy_input, model_specs, DEVICE)
assert _seg_probs.shape == (1, len(CLASSES), HEIGHT, WIDTH), _seg_probs.shape
assert _cls_probs.shape == (1, len(CLASSES)), _cls_probs.shape
assert torch.all((_seg_probs >= 0) & (_seg_probs <= 1))
print("Ensemble + TTA smoke test OK —", _seg_probs.shape, _cls_probs.shape)
del _dummy_input, _seg_probs, _cls_probs

Ensemble + TTA smoke test OK — torch.Size([1, 4, 128, 800]) torch.Size([1, 4])


## Two-threshold gating: conservative gate, looser mask

One threshold doing both jobs (deciding "is anything here" and shaping the mask boundary) forces a trade-off neither job needs. Splitting them: a **gate threshold** on the classification head's probability decides whether to predict anything at all for that class; only once gated in does a separate, looser **mask threshold** shape the actual pixels, recovering fuzzy boundaries and small fragments that a single conservative threshold would otherwise cut off.

Predictions are cached once over the validation set (ensemble + TTA is expensive to rerun per threshold candidate), then both threshold grids are searched against the cached tensors.

In [37]:
def cache_ensemble_predictions(model_specs, loader, device):
    cached = []
    with torch.inference_mode():
        for images, masks in tqdm(loader, desc="Caching ensemble predictions"):
            images = images.to(device, non_blocking=True)
            seg_probs, cls_probs = predict_ensemble_tta(images, model_specs, device)
            cached.append((seg_probs.half().cpu(), cls_probs.half().cpu(), masks.half().cpu()))
    return cached


def select_mask_thresholds(cached_predictions, thresholds, classes, height, width):
    num_t, num_c = len(thresholds), len(classes)
    pixels_per_pair = height * width
    pos_dice_sum = np.zeros((num_t, num_c))
    fp_pixels_sum = np.zeros((num_t, num_c))
    empty_correct_sum = np.zeros((num_t, num_c))
    pos_counts = np.zeros(num_c, dtype=np.int64)
    empty_counts = np.zeros(num_c, dtype=np.int64)
    threshold_tensor = torch.as_tensor(thresholds).view(-1, 1, 1, 1, 1)

    for seg_probs, _, masks in cached_predictions:
        seg_probs, masks = seg_probs.float(), masks.float()
        targets = masks >= 0.5
        target_counts = targets.sum(dim=(-2, -1))
        positive = target_counts > 0
        empty = ~positive
        pos_counts += positive.sum(dim=0).numpy()
        empty_counts += empty.sum(dim=0).numpy()

        predictions = seg_probs.unsqueeze(0) >= threshold_tensor
        predicted_counts = predictions.sum(dim=(-2, -1))
        intersections = (predictions & targets.unsqueeze(0)).sum(dim=(-2, -1))
        pair_dice = 2.0 * intersections.float() / (predicted_counts + target_counts.unsqueeze(0)).clamp_min(1)

        pos_dice_sum += (pair_dice * positive.unsqueeze(0)).sum(dim=1).numpy()
        fp_pixels_sum += (predicted_counts * empty.unsqueeze(0)).sum(dim=1).numpy()
        empty_correct_sum += ((predicted_counts == 0) & empty.unsqueeze(0)).sum(dim=1).numpy()

    contributions = np.zeros((num_t, num_c))
    for c in range(num_c):
        contributions[:, c] = (
            0.70 * pos_dice_sum[:, c] / pos_counts[c]
            - 0.20 * fp_pixels_sum[:, c] / (empty_counts[c] * pixels_per_pair)
            + 0.10 * empty_correct_sum[:, c] / empty_counts[c]
        )
    selected = contributions.argmax(axis=0)
    return {classes[c]: float(thresholds[int(selected[c])]) for c in range(num_c)}, contributions


def select_gate_thresholds(cached_predictions, mask_thresholds, gate_grid, classes, height, width):
    num_g, num_c = len(gate_grid), len(classes)
    pixels_per_pair = height * width
    pos_dice_sum = np.zeros((num_g, num_c))
    fp_pixels_sum = np.zeros((num_g, num_c))
    empty_correct_sum = np.zeros((num_g, num_c))
    pos_counts = np.zeros(num_c, dtype=np.int64)
    empty_counts = np.zeros(num_c, dtype=np.int64)
    mask_threshold_tensor = torch.as_tensor([mask_thresholds[c] for c in classes]).view(1, num_c, 1, 1)
    gate_tensor = torch.as_tensor(gate_grid).view(-1, 1, 1)

    for seg_probs, cls_probs, masks in cached_predictions:
        seg_probs, cls_probs, masks = seg_probs.float(), cls_probs.float(), masks.float()
        targets = masks >= 0.5
        target_counts = targets.sum(dim=(-2, -1))
        positive = target_counts > 0
        empty = ~positive
        pos_counts += positive.sum(dim=0).numpy()
        empty_counts += empty.sum(dim=0).numpy()

        base_pred = seg_probs >= mask_threshold_tensor
        gate_pass = cls_probs.unsqueeze(0) >= gate_tensor
        preds = base_pred.unsqueeze(0) & gate_pass.unsqueeze(-1).unsqueeze(-1)

        predicted_counts = preds.sum(dim=(-2, -1))
        intersections = (preds & targets.unsqueeze(0)).sum(dim=(-2, -1))
        pair_dice = 2.0 * intersections.float() / (predicted_counts + target_counts.unsqueeze(0)).clamp_min(1)

        pos_dice_sum += (pair_dice * positive.unsqueeze(0)).sum(dim=1).numpy()
        fp_pixels_sum += (predicted_counts * empty.unsqueeze(0)).sum(dim=1).numpy()
        empty_correct_sum += ((predicted_counts == 0) & empty.unsqueeze(0)).sum(dim=1).numpy()

    contributions = np.zeros((num_g, num_c))
    for c in range(num_c):
        contributions[:, c] = (
            0.70 * pos_dice_sum[:, c] / pos_counts[c]
            - 0.20 * fp_pixels_sum[:, c] / (empty_counts[c] * pixels_per_pair)
            + 0.10 * empty_correct_sum[:, c] / empty_counts[c]
        )
    selected = contributions.argmax(axis=0)
    return {classes[c]: float(gate_grid[int(selected[c])]) for c in range(num_c)}, contributions

In [38]:
cached_val_predictions = cache_ensemble_predictions(model_specs, val_loader_cnn, DEVICE)

mask_thresholds, mask_contributions = select_mask_thresholds(cached_val_predictions, THRESHOLD_GRID, CLASSES, HEIGHT, WIDTH)
print("Selected mask thresholds:", mask_thresholds)

GATE_GRID = np.array([0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80], dtype=np.float32)
gate_thresholds, gate_contributions = select_gate_thresholds(cached_val_predictions, mask_thresholds, GATE_GRID, CLASSES, HEIGHT, WIDTH)
print("Selected gate thresholds:", gate_thresholds)

for class_name in CLASSES:
    assert gate_thresholds[class_name] not in (GATE_GRID[0], GATE_GRID[-1]), \
        f"{class_name} gate threshold saturated at grid edge — widen GATE_GRID"
print("Gate threshold saturation check passed")

Caching ensemble predictions:   0%|          | 0/285 [00:00<?, ?it/s]

Selected mask thresholds: {'defect_a': 0.25, 'defect_b': 0.30000001192092896, 'defect_c': 0.4000000059604645, 'defect_d': 0.3499999940395355}
Selected gate thresholds: {'defect_a': 0.3499999940395355, 'defect_b': 0.05000000074505806, 'defect_c': 0.30000001192092896, 'defect_d': 0.44999998807907104}


AssertionError: defect_b gate threshold saturated at grid edge — widen GATE_GRID

## Min-area post-processing and final submission

`MIN_COMPONENT_AREA = 20` is conservative by design — your own EDA found `defect_b`'s smallest true connected component at 93px, so a 20px floor only removes single-pixel/speckle noise and shouldn't touch genuine small defects. Validate this against the per-class minimum component sizes you already computed before trusting it blindly on a class you haven't checked.

In [ ]:
MIN_COMPONENT_AREA = 20


def remove_small_components(binary_mask, min_area):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask.astype(np.uint8), connectivity=8)
    cleaned = np.zeros_like(binary_mask, dtype=np.uint8)
    for label in range(1, num_labels):
        if stats[label, cv2.CC_STAT_AREA] >= min_area:
            cleaned[labels == label] = 1
    return cleaned

In [ ]:
def generate_submission(model_specs, mask_thresholds, gate_thresholds, min_area, output_path):
    test_image_ids = sorted(sample_submission["image_id"].astype(str).unique())
    test_dataset = TestDefectDataset(test_image_ids, TEST_IMAGE_DIR, HEIGHT, WIDTH)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

    mask_threshold_tensor = torch.as_tensor([mask_thresholds[c] for c in CLASSES], device=DEVICE).view(1, -1, 1, 1)
    gate_threshold_tensor = torch.as_tensor([gate_thresholds[c] for c in CLASSES], device=DEVICE).view(1, -1)

    records = []
    with torch.inference_mode():
        for image_ids_batch, images in tqdm(test_loader, desc="Generating test predictions"):
            images = images.to(DEVICE, non_blocking=True)
            seg_probs, cls_probs = predict_ensemble_tta(images, model_specs, DEVICE)

            base_pred = (seg_probs >= mask_threshold_tensor).cpu().numpy()
            gate_pass = (cls_probs >= gate_threshold_tensor).cpu().numpy()

            for i, image_id in enumerate(image_ids_batch):
                for c, class_name in enumerate(CLASSES):
                    if gate_pass[i, c]:
                        binary_mask = remove_small_components(base_pred[i, c], min_area)
                    else:
                        binary_mask = np.zeros((HEIGHT, WIDTH), dtype=np.uint8)
                    records.append({
                        "image_id": image_id, "defect_type": class_name,
                        "mask_rle": encode_rle(binary_mask),
                    })

    predictions_df = pd.DataFrame(records)
    submission = sample_submission[["ID", "image_id", "defect_type"]].merge(
        predictions_df, on=["image_id", "defect_type"], how="left"
    )
    submission["mask_rle"] = submission["mask_rle"].fillna("")
    submission.loc[submission["mask_rle"].str.strip() == "", "mask_rle"] = " "

    assert len(submission) == len(sample_submission), "Row count mismatch against sample_submission"
    assert list(submission.columns) == ["ID", "image_id", "defect_type", "mask_rle"], submission.columns.tolist()

    submission.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL)
    print("Non-empty predicted masks:", int((submission["mask_rle"].str.strip() != "").sum()), "/", len(submission))
    return submission


final_submission = generate_submission(
    model_specs, mask_thresholds, gate_thresholds, MIN_COMPONENT_AREA,
    OUTPUT_DIR / "submission_v7_ensemble.csv",
)
final_submission.head(10)

## W&B run comparison

Pulls the logged runs back via the API into a local comparison table — satisfies "at least three runs must be compared using common evaluation metrics" directly, and gives you a ready screenshot for the viva.

In [ ]:
api = wandb.Api()
runs = api.runs(f"{WANDB_ENTITY}/{WANDB_PROJECT}")

comparison_rows = []
for run in runs:
    comparison_rows.append({
        "run_name": run.name,
        "role": run.config.get("role"),
        "pretrained": run.config.get("pretrained"),
        "epochs": run.config.get("epochs"),
        "val_weighted_score": run.summary.get("val_weighted_score"),
        "ema_val_weighted_score": run.summary.get("ema_val_weighted_score"),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("val_weighted_score", ascending=False)
comparison_df

In [ ]:
import pandas as pd

submission_df = final_submission.copy()

submission_df["mask_rle"] = submission_df["mask_rle"].str.strip()

submission_df.to_csv("submission.csv", index=False)

print("Saved to submission.csv successfully!")
print("Empty predictions count:", int((submission_df["mask_rle"] == "").sum()))
submission_df.head(10)

## Requirement checklist

| Requirement | Source | Status after running this notebook |
|---|---|---|
| ≥3 unique models, one scratch, one pretrained, one additional | Guideline §5a, §9 | Model A (scratch), Model B (pretrained), Model C (SegFormer MiT-B4) — Model D (DeepLabV3+) is a bonus fourth |
| Valid W&B runs, ≥3 compared on common metrics | Guideline §5a, §9 | Four runs logged (`train_model` calls `wandb.init`/`wandb.log` per model); pulled into `comparison_df` above |
| Kaggle score ≥ 0.65 cutoff | Guideline §5a, §6 | Not guaranteed by this notebook alone — submit `submission_v7_ensemble.csv` and check the leaderboard; if still short, the blend weights above and the `EPOCHS_*` values are the first things to tune |
| Only trained-model-generated scores count | Guideline §6 | All logic above runs through trained checkpoints — no manual post-hoc score edits |
| GitHub commit history across >3 weeks, no last-minute dump | Guideline §5a, §9 | Not something this notebook can satisfy for you — commit incrementally as you run each phase, not all at once at the end |
| ≥2 of 5 milestones submitted | Guideline §5a | Outside this notebook's scope — confirm your own submission status |
| Individual, own-authored work | Viva Policy | This notebook was written from the earlier consolidated v6 build, general published technique knowledge, and your own already-verified V6 (DeepLabV3+) run — not from any file whose authorship wasn't confirmed as yours |

Everything above was written and reasoned through without GPU or dataset access on this end, so nothing here has been executed against your real data. Run the smoke-test cells first — they use small random tensors and should finish in seconds — before committing to any of the full training cells. If `SEGFORMER_VARIANT = "b4"` doesn't fit your GPU memory even at batch size 1, switch it to `"b0"` — the rest of the notebook doesn't depend on which one you pick.